# 04 -- Collaborative Filtering (implicit ALS)

Matrix-factorization collaborative filtering via the **`implicit`** library
(`implicit.als.AlternatingLeastSquares`), evaluated with **precision@10**.

### Algorithm

Implicit-feedback ALS (Hu, Koren & Volinsky 2008):
1. Build a sparse user x song matrix from play counts; each cell is a
   confidence `c_ui = alpha * play_count`.
2. `AlternatingLeastSquares` alternates least-squares updates to extract
   latent user and song factors that best explain the play signal.
3. **4a -- user-based**: score every song for a user as `U_u . V_i`, recommend
   the top-10 the user hasn't played.
4. **4b -- item-based**: for a seed song, rank songs by cosine similarity of
   their latent profiles, recommend the top-10.

### Evaluation
- **Train/test split**: random holdout of 20% per user (no timestamps in
  the data, so "last 20%" is approximated by a random split).
- **Metric**: Precision@10 -- fraction of recommended tracks the user
  actually listened to in the test set. Target: **> 10%**.

### Output columns
| Column | Description |
|--------|-------------|
| `rank` | 1-10, by likelihood score (descending) |
| `artist` | Artist name |
| `title` | Track title |

In [1]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path.cwd().parent))

from implicit.evaluation import train_test_split

from src.data import MySpotifyRecommender
from src.models.collaborative_filtering import (
    build_user_item_matrix,
    evaluate_user_cf,
    fit_als,
    recommend_tracks_df,
    recommend_users_df,
)


/home/samy/MySpotify/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rs = MySpotifyRecommender.from_files(
    data_dir=Path.cwd().parent / "data",
    download=True,
    # triplets_sample_rows=1_000_000
)

  tracks      (1000000, 4)
  genres      (280831, 3)
  triplets    (48373586, 3)
  lyrics_long (16845943, 3)


---
## Research

### Train / Test Split

In [3]:
user_item, user_idx, song_idx, idx_song = build_user_item_matrix(rs)
user_item.shape

(2018374, 2018374)

In [4]:
train, test = train_test_split(user_item, train_percentage=0.8, random_state=42)

### 4. Collaborative Filtering

In [5]:
model = fit_als(train, factors=192, regularization=0.09, alpha=1.0, iterations=25)

/home/samy/MySpotify/.venv/lib/python3.11/site-packages/implicit/cpu/als.py:96: RuntimeWarning: OpenBLAS is configured to use 28 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
100%|██████████| 25/25 [09:52<00:00, 23.72s/it]


### 4a -- User-based recommendations (latent factors)

for user - b7815dbb206eb2831ce0fe040d0aa537e2e800f7

In [6]:
sample_user = 'b7815dbb206eb2831ce0fe040d0aa537e2e800f7'

print(f"Sample user: {sample_user}")

recommend_users_df(
    sample_user,
    model,
    user_item,
    user_idx,
    idx_song,
    rs.tracks,
    10
)

Sample user: b7815dbb206eb2831ce0fe040d0aa537e2e800f7


index_number,artist_name,track_title,score
u32,str,str,f32
0,"""Tub Ring""","""Invalid""",0.354337
1,"""Cage The Elephant""","""Ain't No Rest For The Wicked (…",0.287669
2,"""Jacky Terrasson""","""Le Jardin d'Hiver""",0.282256
3,"""Eminem""","""Without Me""",0.27861
4,"""Eminem""","""The Real Slim Shady""",0.266597
5,"""Eminem / Nate Dogg""","""'Till I Collapse""",0.262959
6,"""Alliance Ethnik""","""Creil City""",0.239613
7,"""Eminem / Dina Rae""","""Superman""",0.232983
8,"""O'Rosko Raricim""","""Terre Promise""",0.231365


### 4b -- Similar tracks (item-based CF)

In [7]:
top_song = 'SOWYSKH12AF72A303A'

print(f"Top recommended songs for song: {top_song}")

# item_id = song_idx[top_song]

# print(f"\n{rs.tracks.filter(pl.col('song_id') == top_song).select(['artist', 'title']).row(0, named=True)}")

# similar_indices, similarity_scores = model.similar_items(itemid=item_id, N=11)
# similar_indices = similar_indices[1:]
# similarity_scores = similarity_scores[1:]

# similar_item_ids = [idx_song[idx] for idx in similar_indices]

# df = rs.tracks.filter(pl.col("song_id").is_in(similar_item_ids))
# df = df.unique(subset=["song_id"])
# df.select(["artist", "title"])

recommend_tracks_df(
    top_song,
    model,
    song_idx,
    idx_song,
    rs.tracks,
    10
)

Top recommended songs for song: SOWYSKH12AF72A303A


SchemaError: datatypes of join keys don't match - `song_id`: str on left does not match `song_id`: cat on right (and no other type was available to cast to)

#### Evaluation -- Precision@10

Pooled precision@10 over a random sample of test users. Measures what fraction of the top-10
recommended items appear in each user's held-out listening history.

- **ALS params**: factors=192, regularization=0.09, alpha=1.0, iterations=25
- **Train/test split**: 80/20 random
- **Target**: > 10%

In [ ]:
pk_eval = evaluate_user_cf(
    model,
    train,
    test,
)

print(f"Precision@10: {pk_eval:.4f} ({pk_eval*100:.2f}%)")

Precision@10: 0.1151 (11.51%)
